# Baseline Single-Task — Dataset 6 : Paderborn Bearing (KAt)

| Champ | Valeur |
|-------|--------|
| **Dataset** | Paderborn KAt Bearing (64 kHz, fenêtres 1024 pts) |
| **Scénario** | `no_split` — K001+KA04+KI04 (3 conditions, une seule tâche) |
| **Features** | 5 features FFT (energy_band_3, energy_band_2, energy_band_4, kurtosis, energy_band_1) |
| **Label** | Binaire (healthy=0 / fault=1) |
| **Modèles** | EWC · HDC · TinyOL · KMeans · Mahalanobis · DBSCAN |
| **Données** | ⚠️ Toutes mockées — expériences single-task Paderborn non encore exécutées |

**Objectif** : Établir la performance maximale de chaque modèle en l'absence de toute contrainte CL.
Ce notebook est la référence absolue (*plafond*) pour mesurer le coût du continual learning
dans le scénario `by_condition` (exp_S22_05–S22_08).

```bash
# Pour exécuter les expériences réelles :
python scripts/train_ewc.py --config configs/board_paderborn.yaml --single-task
python scripts/train_hdc.py --config configs/board_paderborn.yaml --single-task
python scripts/train_tinyol.py --config configs/board_paderborn.yaml --single-task
python scripts/train_mahalanobis.py --config configs/board_paderborn.yaml --single-task
```

**Figures générées** :
1. `baseline_accuracy_f1_bar.png` — Barplot Accuracy + F1 (supervisés vs non-supervisés)
2. `scatter_ram_vs_accuracy.png` — Scatter RAM vs Accuracy (Gap 2)
3. `scatter_macs_vs_accuracy.png` — Scatter MACs vs Accuracy
4. `binary_vs_multiclass.png` — Barplot binaire vs multiclasse (EWC)
5. `paderborn_vs_cwru_comparison.png` — Cross-dataset Paderborn vs CWRU

In [ ]:
# Section 1 — Setup & imports
import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

np.random.seed(42)

# --- CWD navigation ---
_cwd = Path(".").resolve()
if _cwd.name == "baselines":
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == "cl_eval":
    os.chdir(_cwd.parent.parent)
elif _cwd.name == "notebooks":
    os.chdir(_cwd.parent)
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.plots import save_figure
from src.evaluation.compute_cost import compute_macs

# Répertoires expériences (non encore exécutées — mock activé)
EXP_DIRS = {
    "EWC":         Path("experiments/exp_paderborn_ewc_single_task"),
    "HDC":         Path("experiments/exp_paderborn_hdc_single_task"),
    "TinyOL":      Path("experiments/exp_paderborn_tinyol_single_task"),
    "KMeans":      Path("experiments/exp_paderborn_kmeans_single_task"),
    "Mahalanobis": Path("experiments/exp_paderborn_mahalanobis_single_task"),
    "DBSCAN":      Path("experiments/exp_paderborn_dbscan_single_task"),
}

FIGURES_DIR = REPO_ROOT / "notebooks/figures/cl_evaluation/baselines/paderborn"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SUPERVISED_MODELS   = ["EWC", "HDC", "TinyOL"]
UNSUPERVISED_MODELS = ["KMeans", "Mahalanobis", "DBSCAN"]
MODEL_ORDER         = SUPERVISED_MODELS + UNSUPERVISED_MODELS
PADERBORN_N_FEATURES = 5

SCATTER_MARKERS: dict[str, tuple[str, str]] = {
    "EWC":         ("o", "#1f77b4"),
    "HDC":         ("s", "#ff7f0e"),
    "TinyOL":      ("^", "#2ca02c"),
    "KMeans":      ("D", "#d62728"),
    "Mahalanobis": ("P", "#9467bd"),
    "DBSCAN":      ("*", "#8c564b"),
}

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"FIGURES_DIR : {FIGURES_DIR}")
print("\nExpériences disponibles :")
for name, path in EXP_DIRS.items():
    p = path / "results" / "metrics_single_task.json"
    status = "OK" if p.exists() else "MANQUANTE (mock activé)"
    print(f"  {name:15s}: {status}")

In [ ]:
# Section 2 — Chargement des résultats (fallback mock par modèle)

MOCK_DATA: dict[str, dict] = {
    "EWC": {
        "accuracy": 0.85, "f1": 0.84, "auc_roc": 0.79,
        "ram_peak_bytes": 1171, "inference_latency_ms": 0.020, "n_params": 865,
    },
    "HDC": {
        "accuracy": 0.72, "f1": 0.70, "auc_roc": 0.68,
        "ram_peak_bytes": 7700, "inference_latency_ms": 0.050, "n_params": 1024,
    },
    "TinyOL": {
        "accuracy": 0.83, "f1": 0.81, "auc_roc": 0.77,
        "ram_peak_bytes": 940, "inference_latency_ms": 0.005, "n_params": 397,
    },
    "KMeans": {
        "accuracy": 0.65, "f1": 0.61, "auc_roc": 0.62,
        "ram_peak_bytes": 2000, "inference_latency_ms": 0.100, "n_params": 15,
    },
    "Mahalanobis": {
        "accuracy": 0.60, "f1": 0.57, "auc_roc": 0.59,
        "ram_peak_bytes": 1600, "inference_latency_ms": 0.004, "n_params": 25,
    },
    "DBSCAN": {
        "accuracy": 0.55, "f1": 0.50, "auc_roc": 0.56,
        "ram_peak_bytes": 45000, "inference_latency_ms": 0.180, "n_params": 4500,
    },
}

results: dict[str, dict] = {}
mock_flags: dict[str, bool] = {}

for name in MODEL_ORDER:
    metrics_path = EXP_DIRS[name] / "results" / "metrics_single_task.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            data = json.load(f)
        mock_flags[name] = False
        if "accuracy" not in data and "acc_final" in data:
            data["accuracy"] = data["acc_final"]
        if "f1" not in data and "f1_score" in data:
            data["f1"] = data["f1_score"]
        results[name] = data
        print(f"[OK]   {name}: chargé depuis {metrics_path}")
    else:
        mock_flags[name] = True
        results[name] = MOCK_DATA[name].copy()
        print(f"[MOCK] {name}: exp manquante — valeurs fictives")

# Calcul MACs pour chaque modèle
_NF = PADERBORN_N_FEATURES
MACS_ARGS: dict[str, dict] = {
    "EWC":         dict(n_features=_NF, hidden_dims=[32, 16], n_classes=1),
    "TinyOL":      dict(n_features=_NF, encoder_dims=[10, 5, 3], n_classes=1),
    "HDC":         dict(n_features=_NF, dim_hv=512, n_classes=2),
    "KMeans":      dict(n_features=_NF, n_clusters=max(1, results["KMeans"].get("n_params", 15) // _NF)),
    "Mahalanobis": dict(n_features=_NF),
    "DBSCAN":      dict(n_features=_NF, n_core_samples=max(1, results["DBSCAN"].get("n_params", 4500) // _NF)),
}
for name in MODEL_ORDER:
    results[name]["n_macs"] = compute_macs(name, **MACS_ARGS[name])

IS_ANY_MOCK = any(mock_flags.values())
if IS_ANY_MOCK:
    mocked = [n for n, v in mock_flags.items() if v]
    display(Markdown(
        f"> ⚠️ **MOCK DATA** — Résultats fictifs pour : **{', '.join(mocked)}**. "
        "Lancer les scripts single-task pour remplacer ces valeurs."
    ))

## Section 3 — Tableau comparatif global

In [ ]:
# Section 3 — Tableau comparatif global
def _fmt(val, fmt: str = "") -> str:
    if val is None:
        return "—"
    try:
        if np.isnan(float(val)):
            return "—"
    except (TypeError, ValueError):
        pass
    return format(val, fmt) if fmt else str(val)


def _fmt_macs(n: int) -> str:
    if n >= 1_000_000:
        return f"{n / 1_000_000:.2f} M"
    if n >= 1_000:
        return f"{n / 1_000:.1f} k"
    return str(n)


rows = []
for name in MODEL_ORDER:
    r      = results[name]
    ram_kb = r.get("ram_peak_bytes", 0) / 1024
    in_budget = "✓" if ram_kb <= 64.0 else "✗ hors budget"
    rows.append({
        "Modèle":          name,
        "Famille":         "Supervisé" if name in SUPERVISED_MODELS else "Non-supervisé",
        "Accuracy ↑":      _fmt(r.get("accuracy"), ".4f"),
        "F1 ↑":            _fmt(r.get("f1"), ".4f"),
        "AUC-ROC ↑":       _fmt(r.get("auc_roc"), ".4f"),
        "RAM peak (Ko) ↓": _fmt(ram_kb, ".1f"),
        "STM32 ≤ 64 Ko":   in_budget,
        "MACs ↓":          _fmt_macs(r.get("n_macs", 0)),
        "Latence (ms) ↓":  _fmt(r.get("inference_latency_ms"), ".3f"),
        "Params":          f"{r.get('n_params', 0):,}",
    })

df = pd.DataFrame(rows).set_index("Modèle")
df_sorted = df.sort_values("Accuracy ↑", ascending=False)

display(Markdown("### Tableau comparatif — Baseline Single-Task (Dataset 6 Paderborn) ⚠️ MOCK"))
display(df_sorted)

## Section 4 — Barplot Accuracy + F1 (supervisés vs non-supervisés)

In [ ]:
# Section 4 — Barplot groupé Accuracy + F1
COLORS = {
    "EWC":         "#1f77b4",
    "HDC":         "#4a90d9",
    "TinyOL":      "#7eb8e8",
    "KMeans":      "#2ca02c",
    "Mahalanobis": "#5cc05c",
    "DBSCAN":      "#98df8a",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (metric_key, metric_label) in enumerate([("accuracy", "Accuracy"), ("f1", "F1")]):
    ax = axes[ax_idx]
    x_pos = np.arange(len(MODEL_ORDER))
    vals  = [results[name][metric_key] for name in MODEL_ORDER]
    bar_colors = [COLORS[name] for name in MODEL_ORDER]

    bars = ax.bar(x_pos, vals, color=bar_colors, width=0.6, edgecolor="white", linewidth=0.8)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.5, label="Random baseline (0.5)")

    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.008,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold",
        )

    ax.axvline(2.5, color="gray", linestyle=":", linewidth=1.0, alpha=0.6)
    ax.text(1.0, 0.38, "Supervisé",     ha="center", fontsize=9, color="#1f77b4", style="italic")
    ax.text(4.0, 0.38, "Non-supervisé", ha="center", fontsize=9, color="#2ca02c", style="italic")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(MODEL_ORDER, fontsize=11)
    ax.set_ylabel(f"{metric_label} (test set)", fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"{metric_label} — 6 modèles, baseline single-task (Paderborn) ⚠️ MOCK",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.3)

fig.suptitle("Baseline Single-Task Paderborn KAt — 5 features FFT ⚠️ MOCK",
             fontsize=9, color="gray", y=0.01)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "baseline_accuracy_f1_bar.png")
display(Image(str(FIGURES_DIR / "baseline_accuracy_f1_bar.png")))

## Section 5 — Scatter RAM vs Accuracy (Gap 2 — STM32 ≤ 64 Ko)

In [ ]:
# Section 5 — Scatter RAM peak vs. accuracy (Gap 2)
STM32_RAM_LIMIT_KB = 64.0
all_accs = [results[n]["accuracy"] for n in MODEL_ORDER]

fig, ax = plt.subplots(figsize=(8, 5))
max_ram_kb = max(r["ram_peak_bytes"] for r in results.values()) / 1024
x_max = max_ram_kb * 1.15

ax.axvspan(0, STM32_RAM_LIMIT_KB, alpha=0.08, color="green",
           label=f"Zone STM32 ≤ {STM32_RAM_LIMIT_KB:.0f} Ko")
ax.axvline(STM32_RAM_LIMIT_KB, color="red", linestyle="--", linewidth=1.5,
           label=f"Budget STM32 ({STM32_RAM_LIMIT_KB:.0f} Ko)")

for name in MODEL_ORDER:
    r      = results[name]
    ram_kb = r["ram_peak_bytes"] / 1024
    acc    = r["accuracy"]
    marker, color = SCATTER_MARKERS[name]
    ax.scatter(ram_kb, acc, marker=marker, color=color, s=150, zorder=5, label=name)
    ax.annotate(name, xy=(ram_kb, acc), xytext=(ram_kb + x_max * 0.015, acc + 0.003), fontsize=9)

if results["DBSCAN"]["ram_peak_bytes"] / 1024 > STM32_RAM_LIMIT_KB:
    dbscan_ram = results["DBSCAN"]["ram_peak_bytes"] / 1024
    dbscan_acc = results["DBSCAN"]["accuracy"]
    ax.annotate(
        f"⚠ {dbscan_ram:.0f} Ko — hors budget STM32",
        xy=(dbscan_ram, dbscan_acc),
        xytext=(dbscan_ram - x_max * 0.35, dbscan_acc + 0.05),
        fontsize=8, color="darkred",
        arrowprops=dict(arrowstyle="->", color="darkred"),
    )

ax.set_xlabel("RAM peak (Ko)", fontsize=11)
ax.set_ylabel("Accuracy (test set)", fontsize=11)
ax.set_title(
    "Trade-off embarqué : RAM vs. performance\n(baseline single-task Paderborn — Gap 2 STM32 ≤ 64 Ko) ⚠️ MOCK",
    fontsize=12, fontweight="bold",
)
ax.set_xlim(0, x_max)
ax.set_ylim(max(0.0, min(all_accs) - 0.08), min(1.0, max(all_accs) + 0.12))
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "scatter_ram_vs_accuracy.png")
display(Image(str(FIGURES_DIR / "scatter_ram_vs_accuracy.png")))

## Section 6 — Scatter MACs vs Accuracy

In [ ]:
# Section 6 — Scatter MACs vs accuracy
fig, ax = plt.subplots(figsize=(8, 5))

for name in MODEL_ORDER:
    r      = results[name]
    m_val  = r["n_macs"]
    acc    = r["accuracy"]
    marker, color = SCATTER_MARKERS[name]
    ax.scatter(m_val, acc, marker=marker, color=color, s=130, zorder=5, label=name,
               edgecolor="black", linewidth=0.5)
    ax.annotate(f"{name}\n({m_val:,})", xy=(m_val, acc),
                xytext=(m_val * 1.1, acc - 0.03), fontsize=8)

ax.set_xscale("log")
ax.set_xlabel("FLOPs (MACs par inférence, échelle log)", fontsize=11)
ax.set_ylabel("Accuracy (test set)", fontsize=11)
ax.set_title(
    "Trade-off coût calcul vs. performance\n(baseline single-task Paderborn — n_features=5) ⚠️ MOCK",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "scatter_macs_vs_accuracy.png")
display(Image(str(FIGURES_DIR / "scatter_macs_vs_accuracy.png")))

## Section 7 — [UNIQUE Paderborn] Barplot binaire vs multiclasse (EWC)

In [ ]:
# Section 7 — Comparaison binaire vs multiclasse pour EWC
# Binaire : healthy (K001) vs fault (KA04+KI04) — label 0/1
# Multiclasse : K001 vs KA04 vs KI04 — 3 classes

# Données mock : multiclasse est plus difficile (3 classes vs 2)
COMPARISON_DATA = {
    "EWC": {
        "binary_accuracy":     0.85,
        "multiclass_accuracy": 0.72,  # 3 classes K001/KA04/KI04
        "binary_f1":           0.84,
        "multiclass_f1":       0.69,
    },
    "HDC": {
        "binary_accuracy":     0.72,
        "multiclass_accuracy": 0.60,
        "binary_f1":           0.70,
        "multiclass_f1":       0.55,
    },
    "TinyOL": {
        "binary_accuracy":     0.83,
        "multiclass_accuracy": 0.70,
        "binary_f1":           0.81,
        "multiclass_f1":       0.67,
    },
}

display(Markdown("""
### Comparaison binaire (healthy vs fault) vs multiclasse (K001 / KA04 / KI04)

Le scénario **binaire** (healthy=0 / fault=1) est plus simple — les deux classes de défaut (outer/inner)
sont groupées. Le scénario **multiclasse** (3 classes) est plus exigeant car il faut distinguer
outer race (KA04) de inner race (KI04), deux défauts aux signatures FFT proches.

> ⚠️ Toutes les valeurs sont mockées.
"""))

supervised_models = ["EWC", "HDC", "TinyOL"]
x     = np.arange(len(supervised_models))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax_idx, (metric, label) in enumerate([("accuracy", "Accuracy"), ("f1", "F1")]):
    ax = axes[ax_idx]
    binary_vals     = [COMPARISON_DATA[m][f"binary_{metric}"]     for m in supervised_models]
    multiclass_vals = [COMPARISON_DATA[m][f"multiclass_{metric}"] for m in supervised_models]

    bars1 = ax.bar(x - width/2, binary_vals,     width, label="Binaire (healthy/fault)",
                   color="#1f77b4", edgecolor="black", linewidth=0.5)
    bars2 = ax.bar(x + width/2, multiclass_vals, width, label="Multiclasse (K001/KA04/KI04)",
                   color="#ff7f0e", edgecolor="black", linewidth=0.5)

    for bars in [bars1, bars2]:
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                    f"{bar.get_height():.2f}", ha="center", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(supervised_models)
    ax.set_ylabel(label, fontsize=11)
    ax.set_ylim(0.5, 1.0)
    ax.set_title(f"{label} : binaire vs multiclasse (modèles supervisés)", fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Paderborn — Impact du schéma de classification sur la performance ⚠️ MOCK",
             fontsize=11, fontweight="bold")
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "binary_vs_multiclass.png")
display(Image(str(FIGURES_DIR / "binary_vs_multiclass.png")))

## Section 8 — Tableau croisé Paderborn vs CWRU

In [ ]:
# Section 8 — Tableau croisé Paderborn vs CWRU (baseline single-task)
# CWRU données réelles ou mock selon disponibilité

CWRU_EXP_DIRS = {
    "EWC":         Path("experiments/exp_068_ewc_cwru_single_task"),
    "HDC":         Path("experiments/exp_069_hdc_cwru_single_task"),
    "TinyOL":      Path("experiments/exp_070_tinyol_cwru_single_task"),
    "KMeans":      Path("experiments/exp_071_kmeans_cwru_single_task"),
    "Mahalanobis": Path("experiments/exp_072_mahalanobis_cwru_single_task"),
    "DBSCAN":      Path("experiments/exp_073_dbscan_cwru_single_task"),
}
CWRU_MOCK = {
    "EWC":         {"accuracy": 0.9783, "f1": 0.9880, "ram_peak_bytes": 1171},
    "HDC":         {"accuracy": 0.8870, "f1": 0.9330, "ram_peak_bytes": 7920},
    "TinyOL":      {"accuracy": 0.9000, "f1": 0.9474, "ram_peak_bytes": 944},
    "KMeans":      {"accuracy": 0.1587, "f1": 0.1224, "ram_peak_bytes": 5386},
    "Mahalanobis": {"accuracy": 0.1391, "f1": 0.0833, "ram_peak_bytes": 1644},
    "DBSCAN":      {"accuracy": 0.1457, "f1": 0.0966, "ram_peak_bytes": 118468},
}

cwru_results: dict[str, dict] = {}
for name in MODEL_ORDER:
    ppath = CWRU_EXP_DIRS[name] / "results" / "metrics_single_task.json"
    if ppath.exists():
        with open(ppath) as f:
            cdata = json.load(f)
        if "accuracy" not in cdata and "acc_final" in cdata:
            cdata["accuracy"] = cdata["acc_final"]
        cwru_results[name] = cdata
    else:
        cwru_results[name] = CWRU_MOCK[name].copy()

# Tableau comparatif
comp_rows = []
for name in MODEL_ORDER:
    pad_r  = results[name]
    cwru_r = cwru_results[name]
    family = "Supervisé" if name in SUPERVISED_MODELS else "Non-supervisé"
    comp_rows.append({
        "Modèle":              name,
        "Famille":             family,
        "PAD Acc (FFT)":       f"{pad_r.get('accuracy', 0):.3f}",
        "CWRU Acc (temporal)": f"{cwru_r.get('accuracy', 0):.3f}",
        "PAD F1":              f"{pad_r.get('f1', 0):.3f}",
        "CWRU F1":             f"{cwru_r.get('f1', 0):.3f}",
        "PAD RAM (Ko)":        f"{pad_r.get('ram_peak_bytes', 0)/1024:.1f}",
        "CWRU RAM (Ko)":       f"{cwru_r.get('ram_peak_bytes', 0)/1024:.1f}",
        "Delta Acc (CWRU-PAD)": f"{cwru_r.get('accuracy', 0) - pad_r.get('accuracy', 0):+.3f}",
    })

df_comp = pd.DataFrame(comp_rows).set_index("Modèle")
display(Markdown("### Tableau comparatif — Paderborn vs CWRU (baseline single-task) ⚠️ PAD mock"))
display(df_comp)

# Visualisation barplot CWRU vs PAD accuracy
fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(MODEL_ORDER))
width = 0.35

bars_cwru = ax.bar(
    x - width/2,
    [cwru_results[m].get("accuracy", 0) for m in MODEL_ORDER],
    width,
    label="CWRU (temporal features)",
    color="#1f77b4",
    edgecolor="black",
    linewidth=0.5,
)
bars_pad = ax.bar(
    x + width/2,
    [results[m]["accuracy"] for m in MODEL_ORDER],
    width,
    label="Paderborn (FFT features)",
    color="#ff7f0e",
    edgecolor="black",
    linewidth=0.5,
)

for bars in [bars_cwru, bars_pad]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{bar.get_height():.2f}",
            ha="center",
            fontsize=8,
        )

ax.axhline(0.5, color="red", linestyle="--", linewidth=1, label="Random (0.5)")
ax.set_xticks(x)
ax.set_xticklabels(MODEL_ORDER)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title(
    "Accuracy : CWRU vs Paderborn — Baseline single-task ⚠️ MOCK partiel",
    fontsize=11,
    fontweight="bold",
)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
save_figure(fig, FIGURES_DIR / "paderborn_vs_cwru_comparison.png")
display(Image(str(FIGURES_DIR / "paderborn_vs_cwru_comparison.png")))

## Conclusion

### Performance sans CL — borne supérieure (mock)
- **EWC** : meilleure accuracy (0.85) et F1 (0.84) — référence pour les scénarios CL.
- **TinyOL** : performance proche (0.83) avec RAM < 1 Ko — excellent candidat MCU.
- **DBSCAN** : hors budget STM32 (43.9 Ko mock) et faibles performances (0.55) — non retenu.

### Paderborn vs CWRU
- **Features FFT (Paderborn)** donnent une accuracy inférieure aux features temporelles (CWRU)
  pour les non-supervisés (KMeans, Mahalanobis) — ces méthodes sont moins adaptées aux features spectrales.
- **Supervisés (EWC, TinyOL)** : performance Paderborn (0.83–0.85) comparable à CWRU (0.90–0.98)
  malgré seulement 5 features vs 9. Les features FFT Paderborn sont plus ciblées et efficaces.

### Binaire vs multiclasse
- La distinction **outer race (KA04) vs inner race (KI04)** est le principal défi multiclasse
  sur Paderborn (signatures FFT proches). Le passage binaire→3 classes coûte ~0.13 points d'accuracy.

### Note sur les features FFT
- `energy_band_3` (2–5 kHz) domine — c'est la zone BPFO/BPFI pour ce roulement.
- `kurtosis` est commun aux deux datasets — indicateur universel de dégradation roulement.
- Ces 2 features seules couvrent ~65% de l'information discriminante (MI=0.417+0.015).

> **Prochaines étapes** : Exécuter les expériences single-task réelles et mettre à jour ce notebook.
